# v8 = mezcla de pesos v6 + v7 (sin entrenar)
GPU T4 → **Entorno de ejecución → Ejecutar todo** (~15 min). `clasificador.pt` solo cambia si la mezcla gana sin bajar ningún color.

In [ ]:
!pip install -q "ultralytics>=8.3,<8.4"
# ===== v8 = MEZCLA DE PESOS v6 + v7 ("model soup"), sin entrenar =====
# Idea: v6 es fuerte en plastico/vidrio y v7 en Rojo/organicos. Promediar sus pesos a veces conserva lo mejor de ambos.
# Se prueban 3 mezclas (30%, 50% y 70% de v7). Se elige y calibra con el VAL CONGELADO (2100 fotos que ningun modelo vio).
# clasificador.pt solo cambia si la mezcla gana en val y ningun color baja mas de 2 puntos frente a v6.
from google.colab import drive
drive.mount("/content/drive")
import os, re, shutil, zipfile, hashlib
from datetime import datetime
import numpy as np, pandas as pd, torch
from PIL import Image
from ultralytics import YOLO
from sklearn.metrics import f1_score

DRIVE_DIR = "/content/drive/MyDrive/data basurita"
MODELOS, RUNS, REVISION = f"{DRIVE_DIR}/modelos", f"{DRIVE_DIR}/runs_clasificador", f"{DRIVE_DIR}/revision_v8_mezcla"
MODELO_V5, MODELO_V6 = f"{MODELOS}/clasificador_v5.pt", f"{MODELOS}/clasificador_v6.pt"
V7N = f"{RUNS}/v7n/weights/best.pt"
V6N_RAW = f"{RUNS}/v6n/weights/best.pt"             # v6 sin ajuste (para mezclar pesos)
EXT = (".jpg", ".jpeg", ".png", ".webp", ".bmp")
NOMBRES = {"Azul": "azul_papel_carton", "Amarillo": "amarillo_metales", "Blanco": "blanco_plasticos",
           "Gris": "gris_vidrio", "Marron": "marron_organicos", "Rojo": "rojo_peligrosos", "Negro": "negro_generales"}
LARGO_A_CORTO = {v: k for k, v in NOMBRES.items()}
imagenes = lambda d: sorted(f for f in os.listdir(d) if f.lower().endswith(EXT)) if os.path.isdir(d) else []
for r in (MODELO_V5, MODELO_V6, V7N, V6N_RAW, f"{DRIVE_DIR}/dataset_ntp900058.zip", f"{DRIVE_DIR}/examen_test.zip"):
    assert os.path.exists(r), f"Falta en Drive: {r}"

# ---- 1) Val y test congelados, val en grises, test_camara, examen ----
BASE = "/content/congelado"
shutil.rmtree(BASE, ignore_errors=True)
with zipfile.ZipFile(f"{DRIVE_DIR}/dataset_ntp900058.zip") as z:
    for m in z.namelist():
        partes = m.split("/")
        if len(partes) >= 3 and partes[0] in ("val", "test") and partes[1] in NOMBRES and m.lower().endswith(EXT):
            d = f"{BASE}/{partes[0]}/{NOMBRES[partes[1]]}"
            os.makedirs(d, exist_ok=True)
            open(f"{d}/{os.path.basename(m)}", "wb").write(z.read(m))
VAL_GRIS = "/content/val_gris"
shutil.rmtree(VAL_GRIS, ignore_errors=True)
for c in NOMBRES.values():
    os.makedirs(f"{VAL_GRIS}/{c}", exist_ok=True)
    for f in imagenes(f"{BASE}/val/{c}"):
        Image.open(f"{BASE}/val/{c}/{f}").convert("L").convert("RGB").save(f"{VAL_GRIS}/{c}/{f}")
CAMARA = "/content/test_camara"
shutil.rmtree(CAMARA, ignore_errors=True)
for corto, largo in NOMBRES.items():
    os.makedirs(f"{CAMARA}/{largo}", exist_ok=True)
    for f in imagenes(f"{DRIVE_DIR}/test_camara/{corto}"):
        shutil.copy2(f"{DRIVE_DIR}/test_camara/{corto}/{f}", f"{CAMARA}/{largo}/{f}")
print("val congelado:", {c: len(imagenes(f"{BASE}/val/{c}")) for c in NOMBRES.values()})

GRUPOS_EXAMEN = {
    "cardboard": ("garbage", "Azul (papel/cartón)", {"Azul"}, True),
    "paper": ("garbage", "Azul (papel/cartón)", {"Azul"}, True),
    "metal": ("garbage", "Amarillo (metal)", {"Amarillo"}, True),
    "plastic": ("garbage", "Blanco (plástico)", {"Blanco"}, True),
    "glass": ("garbage", "Gris (vidrio)", {"Gris"}, True),
    "organik": ("garbage_v4", "Marrón (Organik, grises)", {"Marron"}, True),
    "anorganik": ("garbage_v4", "Inorg. reciclable (Anorganik)", {"Amarillo", "Azul", "Blanco", "Gris"}, False),
    "residu": ("garbage_v4", "Negro/Rojo (Residu)", {"Negro", "Rojo"}, False),
    "infectious": ("medico", "Rojo (biocontaminado)", {"Rojo"}, True),
    "general": ("medico", "No-Rojo (médico general)", {"Amarillo", "Azul", "Blanco", "Gris", "Marron", "Negro"}, False)}
shutil.rmtree("/content/examen", ignore_errors=True)
zipfile.ZipFile(f"{DRIVE_DIR}/examen_test.zip").extractall("/content/examen")
filas = []
for raiz, _, archivos in os.walk("/content/examen"):
    clase = os.path.basename(raiz).lower()
    if "test" in raiz.replace("\\", "/").split("/") and clase in GRUPOS_EXAMEN:
        filas += [(os.path.join(raiz, f), *GRUPOS_EXAMEN[clase], clase) for f in archivos if f.lower().endswith(EXT)]
EXAMEN = pd.DataFrame(filas, columns=["ruta", "dataset", "grupo", "colores_ok", "estricta", "clase"])
print(f"Examen: {len(EXAMEN)} imagenes (deben ser 1455)")

# ---- 2) Paso 11: ajuste de los 7 colores y eleccion con el val congelado ----
def probabilidades(ruta_modelo, rutas):
    m = YOLO(ruta_modelo)
    orden = [LARGO_A_CORTO[m.names[i]] for i in range(len(m.names))]
    rutas = list(rutas); P = []
    for i in range(0, len(rutas), 128):
        P += [r.probs.data.cpu().numpy() for r in m.predict(rutas[i:i + 128], imgsz=224, verbose=False, stream=True)]
    return np.array(P), orden

def rutas_de(carpeta):
    rutas, reales = [], []
    for c in sorted(os.listdir(carpeta)):
        for f in imagenes(f"{carpeta}/{c}"):
            rutas.append(f"{carpeta}/{c}/{f}"); reales.append(LARGO_A_CORTO[c])
    return rutas, np.array(reales)

def balanceado(reales, preds):
    return float(np.mean([(preds[reales == c] == c).mean() for c in np.unique(reales)]))

def con_sesgos(P, orden, b):
    return np.array(orden)[(P * np.exp(b)).argmax(1)]

def calibrar(Pc, Pg, orden):
    objetivo = lambda b: (balanceado(VAL_Y, con_sesgos(Pc, orden, b)) + balanceado(VALG_Y, con_sesgos(Pg, orden, b))) / 2
    b = np.zeros(len(orden)); inicial = mejor = objetivo(b)
    for _ in range(3):
        cambio = False
        for k in range(len(orden)):
            for v in np.arange(-1, 2.01, 0.25):
                bb = b.copy(); bb[k] = v
                p = objetivo(bb)
                if p > mejor + 1e-4:
                    mejor, b, cambio = p, bb, True
        if not cambio:
            break
    return b, inicial, mejor

def hornear_sesgos(ruta, b, orden, salida):
    ck = torch.load(ruta, map_location="cpu", weights_only=False)
    m = (ck["ema"] if ck.get("ema") is not None else ck["model"]).float()
    assert [LARGO_A_CORTO[m.names[i]] for i in range(len(m.names))] == orden
    m.model[-1].linear.bias.data += torch.tensor(b, dtype=torch.float32)
    ck["model"], ck["ema"] = m.half(), None
    torch.save(ck, salida)
    return salida

def mezclar(a, salida):
    """Promedia los pesos: (1-a)*v6 + a*v7"""
    c6 = torch.load(V6N_RAW, map_location="cpu", weights_only=False)
    c7 = torch.load(V7N, map_location="cpu", weights_only=False)
    m6 = (c6["ema"] if c6.get("ema") is not None else c6["model"]).float()
    m7 = (c7["ema"] if c7.get("ema") is not None else c7["model"]).float()
    assert m6.names == m7.names
    sd, sd7 = m6.state_dict(), m7.state_dict()
    for k, v in sd.items():
        if v.dtype.is_floating_point:
            sd[k] = (1 - a) * v + a * sd7[k]
    m6.load_state_dict(sd)
    c6["model"], c6["ema"] = m6.half(), None
    torch.save(c6, salida)
    return salida

VAL_R, VAL_Y = rutas_de(f"{BASE}/val")
VALG_R, VALG_Y = rutas_de(VAL_GRIS)
CANDIDATOS = {"v6": MODELO_V6, "v7n": V7N}
for a in (0.3, 0.5, 0.7):
    CANDIDATOS[f"mezcla{int(a * 100)}"] = mezclar(a, f"/content/mezcla{int(a * 100)}.pt")
puntaje, sesgos = {}, {}
for nombre, ruta in list(CANDIDATOS.items()):
    if nombre == "v6":                      # v6 ya trae su ajuste: se evalua tal cual
        Pc, orden = probabilidades(ruta, VAL_R); Pg, _ = probabilidades(ruta, VALG_R)
        puntaje[nombre] = (balanceado(VAL_Y, np.array(orden)[Pc.argmax(1)]) + balanceado(VALG_Y, np.array(orden)[Pg.argmax(1)])) / 2
        print(f"v6    val (color+gris) {puntaje[nombre]:.2%} (entregado, ya calibrado)")
        continue
    Pc, orden = probabilidades(ruta, VAL_R)
    Pg, _ = probabilidades(ruta, VALG_R)
    b, inicial, mejor = calibrar(Pc, Pg, orden)
    puntaje[nombre] = inicial
    print(f"{nombre:<5} val (color+gris) {inicial:.2%} -> con ajuste {mejor:.2%} | ajuste: " +
          (", ".join(f"{c} {v:+.2f}" for c, v in zip(orden, b) if v) or "ninguno"))
    if np.any(b != 0):
        cal = f"{nombre}_cal"
        sesgos[cal] = (b, orden, nombre)
        CANDIDATOS[cal] = hornear_sesgos(ruta, b, orden, f"/content/{cal}.pt")
        puntaje[cal] = mejor
for cal, (b, orden, base) in sesgos.items():
    Pb, _ = probabilidades(CANDIDATOS[base], VAL_R[:200])
    Pc, orden_c = probabilidades(CANDIDATOS[cal], VAL_R[:200])
    print(f"{cal}: el .pt horneado coincide con la simulacion en "
          f"{np.mean(con_sesgos(Pb, orden, b) == np.array(orden_c)[Pc.argmax(1)]):.1%} de 200 fotos")
ELEGIDO = max(puntaje, key=puntaje.get)
print("\nPuntaje en val:", {k: f"{v:.2%}" for k, v in puntaje.items()})
print("Elegido por val:", ELEGIDO)

# ---- 3) Paso 12: test viejo y test_camara ----
def evaluar_carpeta(ruta_modelo, carpeta):
    rutas, reales = rutas_de(carpeta)
    P, orden = probabilidades(ruta_modelo, rutas)
    df = pd.DataFrame({"ruta": rutas, "real": reales, "pred": np.array(orden)[P.argmax(1)]})
    df["ok"] = df.real == df.pred
    return df

def hora(n):
    m = re.search(r"(\d{8}_\d{6})", n)
    return datetime.strptime(m.group(1), "%Y%m%d_%H%M%S") if m else None

def agrupar_por_hora(fotos):
    fotos = sorted(fotos, key=lambda f: (hora(f) or datetime.min, f)); grupos, prev = [], None
    for f in fotos:
        t = hora(f)
        if prev is None or t is None or (t - prev).total_seconds() > 60:
            grupos.append([])
        grupos[-1].append(f); prev = t
    return grupos

def por_objeto(df):
    df = df.assign(nombre=df.ruta.map(os.path.basename)); ok = []
    for clase, g in df.groupby("real"):
        for grupo in agrupar_por_hora(list(g.nombre)):
            ok.append(g[g.nombre.isin(grupo)].pred.mode()[0] == clase)
    return sum(ok), len(ok)

MODELOS_CMP = dict(CANDIDATOS)
R = {(v, p): evaluar_carpeta(ruta, carpeta) for v, ruta in MODELOS_CMP.items()
     for p, carpeta in (("test_viejo", f"{BASE}/test"), ("test_camara", CAMARA))}
print(f"\n{'':<20}" + "".join(f"{v:>10}" for v in MODELOS_CMP))
for p in ("test_viejo", "test_camara"):
    print(f"{p:<20}" + "".join(f"{R[(v, p)].ok.mean():>10.1%}" for v in MODELOS_CMP))
print(f"{'camara (objetos)':<20}" + "".join(f"{'%d/%d' % por_objeto(R[(v, 'test_camara')]):>10}" for v in MODELOS_CMP))

# ---- 4) Paso 13: nota con la formula del docente (solo se MIDE) ----
RANKING_DOCENTE = {"15 Pastor": 16.0, "01 Jhoel": 14.0, "21 Jhosef": 13.6, "08 Fernandez": 13.2,
                   "11 Jorge": 12.9, "02 Cabana": 12.5, "03 Fran": 12.3, "13 Kana": 12.0,
                   "18 Montes": 11.9, "17 Ivan": 11.7, "16 Ciro": 11.2}

def nota_docente(ruta_modelo):
    ex = EXAMEN.copy()
    P, orden = probabilidades(ruta_modelo, ex.ruta)
    ex["pred"] = np.array(orden)[P.argmax(1)]
    ex["ok"] = [p in oks for p, oks in zip(ex.pred, ex.colores_ok)]
    bal = ex.groupby("grupo").ok.mean().mean()
    est = ex[ex.estricta]; y = est.colores_ok.map(lambda s: next(iter(s)))
    mf1 = f1_score(y, est.pred, labels=sorted(y.unique()), average="macro", zero_division=0)
    med = ex[ex.dataset == "medico"]
    rec = (med[med.clase == "infectious"].pred == "Rojo").mean()
    pr = med[med.pred == "Rojo"]; prec = (pr.clase == "infectious").mean() if len(pr) else 0
    f1r = 2 * prec * rec / (prec + rec) if prec + rec else 0
    org = ex[ex.clase == "organik"].ok.mean()
    v1i = ex[ex.dataset == "garbage"]
    tmp = "/content/_v1i_gris"; shutil.rmtree(tmp, ignore_errors=True); os.makedirs(tmp); rut_g = []
    for i, r in enumerate(v1i.ruta):
        d = f"{tmp}/{i}.jpg"; Image.open(r).convert("L").convert("RGB").save(d); rut_g.append(d)
    Pg, og = probabilidades(ruta_modelo, rut_g)
    acc_g = np.mean([p in oks for p, oks in zip(np.array(og)[Pg.argmax(1)], v1i.colores_ok)])
    caida = (v1i.ok.mean() - acc_g) * 100
    seg = 3 * org + 28 / 9 * f1r
    base = 8 * bal + 4 * mf1 + 3.0 + (3 - 0.05 * caida)
    return {"nota": base + seg, "nota_seg_max2": base + min(seg, 2), "exact_bal": bal, "macroF1": mf1,
            "organicos": org, "recall_rojo": rec, "prec_rojo": prec, "caida_gris_pp": caida}, ex

NOTAS = {v: nota_docente(ruta) for v, ruta in MODELOS_CMP.items()}
print("\n", pd.DataFrame({v: NOTAS[v][0] for v in NOTAS}).round(3).to_string())
POR_GRUPO = pd.DataFrame({v: NOTAS[v][1].groupby("grupo").ok.mean() for v in NOTAS})
POR_GRUPO[f"{ELEGIDO} - v6"] = POR_GRUPO[ELEGIDO] - POR_GRUPO["v6"]
print("\nAcierto por grupo del docente:\n", POR_GRUPO.round(3).to_string())
todos = {**RANKING_DOCENTE, **{f"TU {v}": NOTAS[v][0]["nota"] for v in NOTAS}}
print("\nRanking (nota sin recortar):")
for i, (k, n) in enumerate(sorted(todos.items(), key=lambda x: -x[1]), 1):
    print(f"{i:>2}. {k:<14}{n:5.1f}" + ("   <--" if k.startswith("TU") else ""))

# ---- 5) Paso 14: a que color van los errores del elegido ----
ex = NOTAS[ELEGIDO][1]
print(f"\n{ELEGIDO} en el examen: a que color va cada grupo")
print(pd.crosstab(ex.grupo, ex.pred, normalize="index").round(2).to_string())
os.makedirs(REVISION, exist_ok=True)
ex.drop(columns=["colores_ok"]).to_csv(f"{REVISION}/predicciones_examen_{ELEGIDO}.csv", index=False)
POR_GRUPO.to_csv(f"{REVISION}/examen_por_grupo.csv")

# ---- 6) Paso 15: decidir y guardar (misma regla: ningun color baja >2 puntos en val frente a v6) ----
val_c = {v: evaluar_carpeta(CANDIDATOS[v], f"{BASE}/val") for v in dict.fromkeys(("v6", ELEGIDO))}
REC = pd.DataFrame({v: d.groupby("real").ok.mean() for v, d in val_c.items()})
print("\nAcierto por color en val congelado:\n", REC.round(3).to_string())
ok_colores = bool((REC[ELEGIDO] >= REC["v6"] - 0.02).all())
ok_viejo = R[(ELEGIDO, "test_viejo")].ok.mean() >= R[("v6", "test_viejo")].ok.mean() - 0.02
o_e, _ = por_objeto(R[(ELEGIDO, "test_camara")]); o_6, _ = por_objeto(R[("v6", "test_camara")])
ok_cam = o_e >= o_6 - 1
GANA = not ELEGIDO.startswith("v6") and ok_colores and ok_viejo and ok_cam
if not ELEGIDO.startswith("v6"):
    shutil.copy(CANDIDATOS[ELEGIDO], f"{MODELOS}/clasificador_v8.pt")
if GANA:
    shutil.copy(CANDIDATOS[ELEGIDO], f"{MODELOS}/clasificador.pt")
    print(f"\n{ELEGIDO} GANA -> modelos/clasificador.pt ahora es v8 (copia en clasificador_v8.pt)")
else:
    shutil.copy(MODELO_V6, f"{MODELOS}/clasificador.pt")
    print(f"\n{ELEGIDO} no gana (colores ok={ok_colores}, test viejo ok={ok_viejo}, objetos {o_e} vs {o_6})"
          " -> clasificador.pt sigue siendo v6")
print("md5 de clasificador.pt:", hashlib.md5(open(f"{MODELOS}/clasificador.pt", "rb").read()).hexdigest())
REC.to_csv(f"{REVISION}/val_por_color.csv")
for k, ruta in CANDIDATOS.items():
    if k.startswith("mezcla"):
        shutil.copy(ruta, f"{REVISION}/{k}.pt")
print("Evidencias en", REVISION)
